## Executive Summary

**Purpose:** Every diameter, tooth thickness and inspection dimension of an involute spur gear, external or internal, with profile shift.

**What it does:** `GearGeometry` holds one gear (teeth, module, pressure angle, profile shift, basic rack, face width) and derives its reference, base, tip and root diameters, its tooth thickness, the span measurement over *k* teeth and the measurement over two balls or pins. `operating_pressure_angle` and `centre_distance` solve a meshing pair, and `contact_ratio` checks the overlap of the two tooth profiles.

**Units:** gear work is done in millimetres, newtons and MPa throughout `emachines.gears`, because that is what gear standards and gear drawings use. Angles are in radians inside the code and in degrees in reports.

**Sign convention (ISO 21771):** an internal gear is entered with a positive tooth count and `internal=True`. Internally the code gives it a negative tooth count, so one set of formulas covers both kinds of gear. A positive profile shift *x* always adds material to the tooth: an external tooth gets thicker at its base, an internal tooth gets thicker and its tip moves toward the centre.

## How It Fits Into the Gear Design Workflow

**Upstream:** nothing; this is the base of the `gears` module.

**Downstream:** `gears.layout` (tooth counts, centre distance and profile shift of a planetary set), `gears.rating` (ISO 6336 strength), `gears.manufacturing` (gear data sheets and tolerances).

In [ ]:
#| default_exp gears.involute

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, replace

import numpy as np

## Notation

| Symbol | Meaning | Units |
|---|---|---|
| $z$ | number of teeth (negative for an internal gear inside the formulas) | – |
| $m$ | module | mm |
| $\alpha$ | reference (tool) pressure angle | rad |
| $x$ | profile shift coefficient | – |
| $h_{aP}^*,\ h_{fP}^*,\ \rho_{fP}^*$ | basic rack addendum, dedendum and root radius, per unit module | – |
| $d = z m$ | reference diameter | mm |
| $d_b = d \cos\alpha$ | base diameter | mm |
| $d_a,\ d_f$ | tip and root diameters | mm |
| $s$ | tooth thickness on the reference circle (arc) | mm |
| $W_k$ | span measurement over $k$ teeth (base tangent length) | mm |
| $M_d$ | dimension over (external) or between (internal) two balls of diameter $D_M$ | mm |
| $\alpha_w$ | operating pressure angle of a meshing pair | rad |
| $a_w$ | operating centre distance | mm |
| $\varepsilon_\alpha$ | transverse contact ratio | – |

## The Involute Function

$$\operatorname{inv}\alpha = \tan\alpha - \alpha .$$

It is the polar angle of a point on the involute whose pressure angle is $\alpha$, so tooth thicknesses at different diameters are related through it. The inverse has no closed form; `inv_inverse` solves it by Newton's method from Cheng's starting value.

In [ ]:
#| export
def inv(alpha):
    "Involute function tan(α) − α (α in radians; works on arrays)."
    return np.tan(alpha) - alpha


def inv_inverse(value: float, tol: float = 1e-14) -> float:
    "The pressure angle α [rad] whose involute is `value` (0 ≤ value < ~1.5)."
    if value < 0:
        raise ValueError("involute is non-negative for 0 ≤ α < π/2")
    if value == 0:
        return 0.0
    a = (3.0 * value) ** (1.0 / 3.0)           # small-angle start: inv α ≈ α³/3
    a = min(a, 1.4)
    for _ in range(60):
        f = math.tan(a) - a - value
        da = f / math.tan(a) ** 2               # d(inv α)/dα = tan² α
        a -= da
        if abs(da) < tol:
            return a
    raise RuntimeError("inv_inverse did not converge")

In [ ]:
for deg in (14.5, 20.0, 25.0, 31.7):
    a = math.radians(deg)
    assert abs(inv_inverse(float(inv(a))) - a) < 1e-12
print(f"inv 20° = {inv(math.radians(20)):.6f}")   # 0.014904
assert abs(inv(math.radians(20)) - 0.0149044) < 1e-6

## Standard Modules

ISO 54 lists preferred modules (series I) and second-choice modules (series II). Gear cutting tools are stocked in these sizes, so a design should land on one of them.

In [ ]:
#| export
ISO54_SERIES_I = (0.1, 0.12, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0, 25.0, 32.0, 40.0, 50.0)
ISO54_SERIES_II = (0.11, 0.14, 0.18, 0.22, 0.28, 0.35, 0.45, 0.55, 0.7, 0.9, 1.125, 1.375, 1.75, 2.25,
                   2.75, 3.5, 4.5, 5.5, 7.0, 9.0, 11.0, 14.0, 18.0, 22.0, 28.0, 36.0, 45.0)


def standard_modules(m_min: float = 0.0, m_max: float = 1e9, series_ii: bool = False) -> list[float]:
    "ISO 54 modules in [m_min, m_max], series I only unless `series_ii`."
    ms = ISO54_SERIES_I + (ISO54_SERIES_II if series_ii else ())
    return sorted(m for m in ms if m_min - 1e-12 <= m <= m_max + 1e-12)

In [ ]:
assert standard_modules(0.9, 2.1) == [1.0, 1.25, 1.5, 2.0]
assert 1.75 in standard_modules(1.6, 1.8, series_ii=True)

## `GearGeometry`: Theory

A spur gear is fully defined by $z$, $m$, $\alpha$, $x$, the basic rack of the cutting tool, and the face width $b$ (ISO 21771). With the signed tooth count ($z<0$ for an internal gear, so $d<0$ too):

$$d = z m,\qquad d_b = d\cos\alpha,\qquad p = \pi m,\qquad p_b = p\cos\alpha$$

$$d_a = d + 2m\,(h_{aP}^* + x - k^*),\qquad d_f = d - 2m\,(h_{fP}^* - x)$$

where $k^*$ is an optional tip reduction (for an internal gear it is used to keep the ring tips clear of the planet roots and to lift them off the base circle). For an internal gear the formulas give negative diameters; their sizes are the diameters you would measure.

Arc tooth thickness on the reference circle, and on any other circle of diameter $d_y$ with pressure angle $\alpha_y = \arccos(d_b/d_y)$:

$$s = m\left(\frac{\pi}{2} + 2x\tan\alpha\right),\qquad s_y = d_y\left(\frac{s}{d} + \operatorname{inv}\alpha - \operatorname{inv}\alpha_y\right).$$

The tip thickness $s_a$ (tooth thickness on the tip circle) must stay above about $0.25\,m$ for case-hardened teeth, otherwise the tip through-hardens and chips.

**Undercut.** A rack-generated external gear is undercut (the cutter removes the bottom of the working involute) when

$$z < z_{min} = \frac{2\,(h_{aP0}^* - x - \rho_{aP0}^*(1-\sin\alpha))}{\sin^2\alpha},$$

with $h_{aP0}^* = h_{fP}^*$ the cutter addendum and $\rho_{aP0}^* = \rho_{fP}^*$ its tip radius. For the ISO 53 profile A rack ($\alpha = 20°$, $h_{fP}^* = 1.25$, $\rho_{fP}^* = 0.38$) this is 17 teeth at $x = 0$.

In [ ]:
#| export
@dataclass(frozen=True)
class GearGeometry:
    """
    One involute spur gear (ISO 21771). Lengths in mm, angles in radians.

    Attributes
    ----------
    z        : int    Number of teeth, always entered positive
    m        : float  Module [mm]
    x        : float  Profile shift coefficient (positive adds material to the tooth, ISO 21771)
    b        : float  Face width [mm]
    alpha    : float  Reference pressure angle [rad] (default 20°)
    internal : bool   True for an internal (ring) gear
    ha       : float  Basic rack addendum coefficient h_aP* (default 1.0)
    hf       : float  Basic rack dedendum coefficient h_fP* (default 1.25)
    rho_f    : float  Basic rack root radius coefficient ρ_fP* (default 0.38, ISO 53 profile A)
    k_tip    : float  Tip reduction coefficient k* (default 0)
    name     : str    Label used in reports
    """
    z: int
    m: float
    x: float = 0.0
    b: float = 10.0
    alpha: float = math.radians(20.0)
    internal: bool = False
    ha: float = 1.0
    hf: float = 1.25
    rho_f: float = 0.38
    k_tip: float = 0.0
    name: str = "gear"

    def __post_init__(self):
        if self.z < 3:
            raise ValueError("z must be a positive tooth count ≥ 3 (use internal=True for a ring gear)")
        if self.m <= 0 or self.b <= 0:
            raise ValueError("module and face width must be positive")

    # --- signed quantities (ISO 21771: negative for internal gears) ---------
    @property
    def zs(self) -> int:
        "Signed tooth count (negative for an internal gear)."
        return -self.z if self.internal else self.z

    @property
    def d_signed(self) -> float: return self.zs * self.m

    # --- diameters (always positive) -----------------------------------------
    @property
    def d(self) -> float:
        "Reference diameter [mm]."
        return abs(self.d_signed)

    @property
    def db(self) -> float:
        "Base diameter [mm]."
        return self.d * math.cos(self.alpha)

    @property
    def da(self) -> float:
        "Tip diameter [mm] (for an internal gear: the tip circle inside the ring)."
        add = 2 * self.m * (self.ha + self.x - self.k_tip)
        return self.d - add if self.internal else self.d + add

    @property
    def df(self) -> float:
        "Root diameter [mm] (for an internal gear: the root circle at the outside of the teeth)."
        ded = 2 * self.m * (self.hf - self.x)
        return self.d + ded if self.internal else self.d - ded

    @property
    def p(self) -> float: return math.pi * self.m

    @property
    def pb(self) -> float: return self.p * math.cos(self.alpha)

    @property
    def h(self) -> float:
        "Tooth depth (tip to root) [mm]."
        return abs(self.df - self.da) / 2

    # --- tooth thickness -------------------------------------------------------
    @property
    def s(self) -> float:
        "Arc tooth thickness on the reference circle [mm], no allowance."
        return self.m * (math.pi / 2 + 2 * self.x * math.tan(self.alpha))

    def s_at(self, dy: float, s: float | None = None) -> float:
        "Arc tooth thickness on diameter `dy` [mm], given reference thickness `s` (default: nominal)."
        s = self.s if s is None else s
        if dy < self.db - 1e-9:
            raise ValueError("diameter is inside the base circle: no involute there")
        ay = math.acos(min(1.0, self.db / dy))
        if self.internal:   # the internal tooth widens toward its root (larger diameter)
            return dy * (s / self.d - inv(self.alpha) + inv(ay))
        return dy * (s / self.d + inv(self.alpha) - inv(ay))

    @property
    def sa(self) -> float:
        "Tooth thickness on the tip circle [mm]."
        return self.s_at(max(self.da, self.db))

    @property
    def z_min_undercut(self) -> float:
        "Smallest tooth count without undercut for this profile shift (rack-generated external gear)."
        a = self.alpha
        return 2 * (self.hf - self.x - self.rho_f * (1 - math.sin(a))) / math.sin(a) ** 2

    @property
    def undercut(self) -> bool:
        "True if a rack-generated external gear with this shift is undercut."
        return (not self.internal) and self.z < self.z_min_undercut - 1e-9

    @property
    def d_form(self) -> float:
        """
        Root form diameter d_Ff [mm]: where the generated involute meets the fillet.

        Exact for a rack-generated (hobbed) external gear. For an internal gear the
        mirrored rack formula is used; a pinion-type shaper cutter gives a slightly
        different fillet, so treat it as an estimate and confirm it with the cutter data.
        """
        a = self.alpha
        h_lin = self.m * (self.hf - self.rho_f * (1 - math.sin(a)) - self.x)   # straight-flank depth below pitch line
        if self.internal:
            g = self.d * math.sin(a) / 2 + h_lin / math.sin(a)
        else:
            g = self.d * math.sin(a) / 2 - h_lin / math.sin(a)
        if g < 0:   # generated involute stops inside the base circle: undercut
            return self.db
        return 2 * math.hypot(self.db / 2, g)

    def with_(self, **kw) -> "GearGeometry":
        "A copy with some fields changed."
        return replace(self, **kw)

### Example

A 17-tooth, module 1.5 sun with a +0.3 shift and the 67-tooth ring of the same planetary set.

In [ ]:
sun = GearGeometry(z=17, m=1.5, x=0.3, b=20, name="sun")
ring = GearGeometry(z=67, m=1.5, x=0.0, b=20, internal=True, name="ring")
for g in (sun, ring):
    print(f"{g.name:5s} d = {g.d:7.3f}  db = {g.db:7.3f}  da = {g.da:7.3f}  df = {g.df:7.3f}  "
          f"s = {g.s:.4f}  sa = {g.sa:.3f}  d_Ff = {g.d_form:.3f} mm")
assert math.isclose(sun.da, 25.5 + 2 * 1.5 * 1.3) and math.isclose(sun.df, 25.5 - 2 * 1.5 * 0.95)
assert math.isclose(ring.da, 100.5 - 3.0) and math.isclose(ring.df, 100.5 + 3.75)
assert not sun.undercut and GearGeometry(z=12, m=1).undercut
assert abs(GearGeometry(z=17, m=1).z_min_undercut - 17.1) < 0.2     # ISO profile A, x = 0

## Inspection Dimensions: Theory

Tooth thickness cannot be measured directly, so gear drawings specify it through a dimension the gear shop can measure: the span over $k$ teeth or the dimension over two balls (or pins). Both are linked exactly to the reference tooth thickness $s$.

**Span measurement (external gears).** A disc micrometer touching two flanks $k$ teeth apart lies along a base tangent, so

$$W_k = (k-1)\,p_b + s_b,\qquad s_b = d_b\left(\frac{s}{d} + \operatorname{inv}\alpha\right).$$

$k$ is chosen so that the anvils touch near the middle of the tooth depth:

$$k = \operatorname{round}\left[\frac{z}{\pi}\left(\tan\alpha_x - \frac{2x\tan\alpha}{z} - \operatorname{inv}\alpha\right) + 0.5\right],\qquad \cos\alpha_x = \frac{d_b}{d + 2xm}.$$

**Dimension over balls (external) or between balls (internal).** A ball of diameter $D_M$ in a tooth space has its centre at pressure angle $\alpha_M$ with (signed $z$, $d$, $d_b$)

$$\operatorname{inv}\alpha_M = \frac{s}{d} + \operatorname{inv}\alpha + \frac{D_M}{d_b} - \frac{\pi}{z}$$

and the measurement across two balls is

$$M_d = \frac{d_b}{\cos\alpha_M} + D_M\ \ (z \text{ even}),\qquad M_d = \frac{d_b\cos(\pi/2z)}{\cos\alpha_M} + D_M\ \ (z \text{ odd}).$$

For an internal gear the same formulas with negative $z$, $d$, $d_b$ give $M_d = d_b/\cos\alpha_M - D_M$ in positive sizes. The ball touches the flank where $\tan\alpha_c = \tan\alpha_M - D_M/d_b$. `ball_diameter` picks the ball that touches on the reference circle: with $\eta = \pi/|z| - s/d$ the half-angle of the tooth space,

$$D_M = d_b\,[\tan(\alpha + \eta) - \tan\alpha]\ \text{(external)},\qquad D_M = d_b\,[\tan\alpha - \tan(\alpha - \eta)]\ \text{(internal)},$$

rounded to 0.01 mm, a step that measuring-pin sets come in. If the balls would then sit below the tips (common with a positive shift, which raises the tip of an external gear, and on internal gears), the ball is enlarged until it stands out past the tips by $0.2\,m$, so the micrometer touches the balls and not the teeth.

In [ ]:
#| export
def span_teeth(g: GearGeometry) -> int:
    "Number of teeth k to span for the base tangent length of an external gear."
    if g.internal:
        raise ValueError("span measurement is for external gears; use ball_measurement for internal gears")
    a = g.alpha
    ax = math.acos(min(1.0, g.db / (g.d + 2 * g.x * g.m)))
    k = round(g.z / math.pi * (math.tan(ax) - 2 * g.x * math.tan(a) / g.z - inv(a)) + 0.5)
    return int(min(max(k, 2), g.z - 1))


def span_measurement(g: GearGeometry, s: float | None = None, k: int | None = None) -> float:
    "Base tangent length W_k [mm] over k teeth for reference tooth thickness s (default nominal)."
    k = span_teeth(g) if k is None else k
    s = g.s if s is None else s
    sb = g.db * (s / g.d + inv(g.alpha))
    return (k - 1) * g.pb + sb


def ball_diameter(g: GearGeometry, protrude: float = 0.2) -> float:
    """
    Ball or pin diameter [mm], rounded to 0.01 mm: touches the flanks near the reference circle,
    enlarged if needed so the balls stand out past the tips by at least `protrude`·m
    (otherwise a micrometer anvil would rest on the tips instead of the balls).
    """
    eta = math.pi / g.z - g.s / g.d
    a = g.alpha
    if g.internal:
        D = g.db * (math.tan(a) - math.tan(a - eta))
    else:
        D = g.db * (math.tan(a + eta) - math.tan(a))
    D = round(D, 2)
    for _ in range(400):
        M, _ = ball_measurement(g, D)
        clear = (g.da - M) if g.internal else (M - g.da)
        if clear >= protrude * g.m:
            break
        D = round(D + 0.01, 2)
    return D


def ball_measurement(g: GearGeometry, D: float | None = None, s: float | None = None) -> tuple[float, float]:
    """
    Dimension over two balls (external) or between two balls (internal) [mm].

    Returns (M_d, d_contact): the measurement and the diameter where the ball touches the flank.
    """
    D = ball_diameter(g) if D is None else D
    s = g.s if s is None else s
    sg = -1.0 if g.internal else 1.0
    z, d, db = sg * g.z, sg * g.d, sg * g.db
    aM = inv_inverse(s / d + float(inv(g.alpha)) + D / db - math.pi / z)
    factor = 1.0 if g.z % 2 == 0 else math.cos(math.pi / (2 * g.z))
    M = abs(db * factor / math.cos(aM) + D)
    tan_c = math.tan(aM) - D / db
    d_contact = abs(db) / math.cos(math.atan(tan_c))
    return M, d_contact

In [ ]:
# Check against the span tables: z = 30, x = 0 → k = 4, W* = 10.7526 per unit module
g30 = GearGeometry(z=30, m=2)
assert span_teeth(g30) == 4 and abs(span_measurement(g30) / 2 - 10.7526) < 1e-4
# Ball touching at the reference circle
for g in (sun, ring, g30, GearGeometry(z=25, m=1, x=-0.2)):
    D = ball_diameter(g)
    M, dc = ball_measurement(g, D)
    print(f"{g.name:5s} z={g.z:3d}  D_M = {D:.2f}  M_d = {M:.4f}  tip d_a = {g.da:.3f}  contact at d = {dc:.3f} (ref {g.d:.3f})")
    assert (g.da - M if g.internal else M - g.da) >= 0.2 * g.m - 1e-9
    assert min(g.da, g.d_form) - 1e-9 <= dc <= max(g.da, g.d_form) + 1e-9      # contact on the working flank

A thinner tooth (the tolerance allowances of `gears.manufacturing`) must give a smaller span and a smaller dimension over balls on an external gear, and a *larger* dimension between balls on an internal gear:

In [ ]:
ds = -0.05
assert span_measurement(sun, sun.s + ds) < span_measurement(sun)
assert ball_measurement(sun, 2.8, sun.s + ds)[0] < ball_measurement(sun, 2.8)[0]
assert ball_measurement(ring, 2.5, ring.s + ds)[0] > ball_measurement(ring, 2.5)[0]
# span change = Δs·cos α exactly
assert math.isclose(span_measurement(sun) - span_measurement(sun, sun.s + ds), -ds * math.cos(sun.alpha))

## Meshing Pair: Theory

Two gears with profile shifts $x_1, x_2$ mesh without backlash at the operating pressure angle (signed $z$ for an internal gear)

$$\operatorname{inv}\alpha_w = \operatorname{inv}\alpha + 2\tan\alpha\,\frac{x_1 + x_2}{z_1 + z_2}$$

and the operating centre distance is

$$a_w = \frac{(z_1 + z_2)\,m}{2}\,\frac{\cos\alpha}{\cos\alpha_w}.$$

Turned around, a required centre distance fixes the shift sum: $\cos\alpha_w = a\cos\alpha / a_w$ with $a = (z_1+z_2)m/2$, then $x_1 + x_2 = (z_1+z_2)(\operatorname{inv}\alpha_w - \operatorname{inv}\alpha)/(2\tan\alpha)$. This is how the planet–ring shift is found once the sun–planet mesh has fixed $a_w$.

**Transverse contact ratio** (gear 1 external, gear 2 external or internal):

$$\varepsilon_\alpha = \frac{\sqrt{r_{a1}^2 - r_{b1}^2} \pm \sqrt{r_{a2}^2 - r_{b2}^2} \mp a_w\sin\alpha_w}{p_b},$$

upper signs for an external pair, lower signs for an internal pair. Keep $\varepsilon_\alpha \ge 1.2$; values from 1.4 to 1.8 are usual for spur gears.

**Interference.** Contact must start above the root form circle $d_{Ff}$ of each gear. On the line of action, with $T_1$, $T_2$ the tangent points on the base circles, the start of active profile is found from the mating tip. For an internal ring the ring tip must also lie beyond $T_1$ ($\sqrt{r_{a2}^2-r_{b2}^2} \ge a_w \sin\alpha_w$), otherwise it digs into the planet root.

**Tip clearance.** The gap between each tip circle and the mating root circle should be at least $0.1\,m$ (0.25 m is the nominal value with the standard rack); below that, oil is trapped and the tip may touch the fillet.

In [ ]:
#| export
def operating_pressure_angle(g1: GearGeometry, g2: GearGeometry) -> float:
    "Operating pressure angle α_w [rad] of a backlash-free mesh (g2 may be internal)."
    if g1.internal:
        raise ValueError("g1 must be the external gear of the pair")
    _check_pair(g1, g2)
    zsum = g1.zs + g2.zs
    return inv_inverse(float(inv(g1.alpha)) + 2 * math.tan(g1.alpha) * (g1.x + g2.x) / zsum)


def centre_distance(g1: GearGeometry, g2: GearGeometry) -> float:
    "Operating centre distance a_w [mm] of a backlash-free mesh."
    aw = operating_pressure_angle(g1, g2)
    return abs((g1.zs + g2.zs) * g1.m / 2 * math.cos(g1.alpha) / math.cos(aw))


def shift_sum_for_centre_distance(z1: int, z2_signed: int, m: float, a_w: float,
                                  alpha: float = math.radians(20.0)) -> float:
    "Profile shift sum x1 + x2 that gives centre distance a_w [mm] (z2 negative for an internal gear)."
    zsum = z1 + z2_signed
    a = abs(zsum) * m / 2
    c = a * math.cos(alpha) / a_w
    if not 0 < c < 1:
        raise ValueError("centre distance not reachable with an involute mesh")
    aw = math.acos(c)
    return zsum * (float(inv(aw)) - float(inv(alpha))) / (2 * math.tan(alpha))


def contact_ratio(g1: GearGeometry, g2: GearGeometry) -> float:
    "Transverse contact ratio ε_α of the pair (g1 external, g2 external or internal)."
    aw = operating_pressure_angle(g1, g2)
    a_w = centre_distance(g1, g2)
    t1 = math.sqrt(max(0.0, (g1.da / 2) ** 2 - (g1.db / 2) ** 2))
    t2 = math.sqrt(max(0.0, (g2.da / 2) ** 2 - (g2.db / 2) ** 2))
    if g2.internal:
        return (t1 - t2 + a_w * math.sin(aw)) / g1.pb
    return (t1 + t2 - a_w * math.sin(aw)) / g1.pb


@dataclass(frozen=True)
class MeshCheck:
    """
    Geometry checks of one mesh. Lengths in mm.

    alpha_w        : operating pressure angle [rad]
    a_w            : operating centre distance
    eps_alpha      : transverse contact ratio
    d_Nf1, d_Nf2   : start of active profile on each gear (g2: for an internal gear, the outermost contact diameter)
    d_Ff1, d_Ff2   : root form diameters
    clearance12    : g1 tip to g2 root
    clearance21    : g2 tip to g1 root
    interference   : True if contact runs into a fillet or (internal) the ring tip lies inside T1
    """
    alpha_w: float
    a_w: float
    eps_alpha: float
    d_Nf1: float
    d_Nf2: float
    d_Ff1: float
    d_Ff2: float
    clearance12: float
    clearance21: float
    interference: bool

    @property
    def ok(self) -> bool:
        m_ok = self.eps_alpha >= 1.2 and not self.interference
        return m_ok and min(self.clearance12, self.clearance21) > 0


def mesh_check(g1: GearGeometry, g2: GearGeometry) -> MeshCheck:
    "Contact ratio, active profile, interference and tip clearance of a mesh."
    aw = operating_pressure_angle(g1, g2)
    a_w = centre_distance(g1, g2)
    rb1, rb2 = g1.db / 2, g2.db / 2
    ra1, ra2 = g1.da / 2, g2.da / 2
    if ra2 < rb2 - 1e-9 and g2.internal:
        raise ValueError(f"{g2.name}: tip circle inside the base circle; raise k_tip")
    t1 = math.sqrt(max(0.0, ra1 ** 2 - rb1 ** 2))
    t2 = math.sqrt(max(0.0, ra2 ** 2 - rb2 ** 2))
    T = a_w * math.sin(aw)                        # T1T2
    if g2.internal:
        g_1 = t2 - T                              # ring tip, measured from T1
        g_2 = T + t1                              # planet tip, measured from T2
        d_Nf1 = 2 * math.hypot(rb1, g_1)
        d_Nf2 = 2 * math.hypot(rb2, g_2)
        interf = g_1 < 0 or d_Nf1 < g1.d_form - 1e-6 or d_Nf2 > g2.d_form + 1e-6
        c12 = g2.df / 2 - a_w - ra1
        c21 = ra2 - a_w - g1.df / 2
    else:
        g_1 = T - t2
        g_2 = T - t1
        d_Nf1 = 2 * math.hypot(rb1, max(g_1, 0.0))
        d_Nf2 = 2 * math.hypot(rb2, max(g_2, 0.0))
        interf = g_1 < 0 or g_2 < 0 or d_Nf1 < g1.d_form - 1e-6 or d_Nf2 < g2.d_form - 1e-6
        c12 = a_w - ra1 - g2.df / 2
        c21 = a_w - ra2 - g1.df / 2
    return MeshCheck(aw, a_w, contact_ratio(g1, g2), d_Nf1, d_Nf2, g1.d_form, g2.d_form, c12, c21, interf)


def _check_pair(g1: GearGeometry, g2: GearGeometry):
    if not math.isclose(g1.m, g2.m) or not math.isclose(g1.alpha, g2.alpha):
        raise ValueError("gears of a pair need the same module and pressure angle")

### Example

The sun above with a 25-tooth planet at $x = -0.3$ (a zero-sum pair, so $a_w = a$), and the ring meshing with the planet at the same centre distance. The ring then needs $x = +0.3$, and its lengthened teeth reach into the planet fillet; shortening them by $0.2\,m$ (`k_tip`) clears it at the cost of some contact ratio.

In [ ]:
planet = GearGeometry(z=25, m=1.5, x=-0.3, b=20, name="planet")
sp = mesh_check(sun, planet)
xr = shift_sum_for_centre_distance(planet.z, -ring.z, 1.5, sp.a_w) - planet.x
pr0 = mesh_check(planet, ring.with_(x=xr))
print(f"ring at x = {xr:+.2f}, no tip reduction: interference = {pr0.interference} "
      f"(contact starts at d = {pr0.d_Nf1:.3f} on the planet, its form circle is {pr0.d_Ff1:.3f})")
ring2 = ring.with_(x=xr, k_tip=0.2)      # shorten the ring teeth by 0.2 m
pr = mesh_check(planet, ring2)
print(f"sun–planet:  a_w = {sp.a_w:.4f}  α_w = {math.degrees(sp.alpha_w):.3f}°  ε_α = {sp.eps_alpha:.3f}  "
      f"clearances {sp.clearance12:.3f}/{sp.clearance21:.3f} mm  ok={sp.ok}")
print(f"planet–ring: a_w = {pr.a_w:.4f}  α_w = {math.degrees(pr.alpha_w):.3f}°  ε_α = {pr.eps_alpha:.3f}  "
      f"clearances {pr.clearance12:.3f}/{pr.clearance21:.3f} mm  x_ring = {xr:+.3f}  ok={pr.ok}")
assert math.isclose(sp.a_w, (17 + 25) * 1.5 / 2) and math.isclose(pr.a_w, sp.a_w)
assert math.isclose(xr, 0.3) and pr0.interference and sp.ok and pr.ok

---

## Tests

In [ ]:
#| hide
# Standard external pair, x = 0: α_w = α and a_w = a
g1, g2 = GearGeometry(z=20, m=2), GearGeometry(z=40, m=2)
assert math.isclose(operating_pressure_angle(g1, g2), math.radians(20)) and math.isclose(centre_distance(g1, g2), 60)
# Contact ratio of 20/40 standard pair (textbook ≈ 1.64)
assert abs(contact_ratio(g1, g2) - 1.64) < 0.01
# Positive shift sum increases centre distance and α_w; shift_sum_for_centre_distance inverts it
g1s, g2s = g1.with_(x=0.3), g2.with_(x=0.2)
aw = centre_distance(g1s, g2s)
assert aw > 60 and math.isclose(shift_sum_for_centre_distance(20, 40, 2, aw), 0.5, abs_tol=1e-9)
# Internal: x = 0 on both gives a = (zr − zp) m / 2
gr = GearGeometry(z=60, m=2, internal=True)
assert math.isclose(centre_distance(g1, gr), 40.0)
# Internal contact ratio is higher than external for the same pinion
assert contact_ratio(g1, gr) > contact_ratio(g1, g2)
# Thickness on reference circle via s_at equals s
for g in (sun, ring2, planet):
    assert math.isclose(g.s_at(g.d), g.s, rel_tol=1e-12)
# Odd tooth count over-ball formula reduces to even formula for large z
gE, gO = GearGeometry(z=200, m=1), GearGeometry(z=201, m=1)
assert abs(ball_measurement(gO, 1.7)[0] - ball_measurement(gE, 1.7)[0] - 1.0) < 0.02
# A mesh with too little clearance is flagged
bad = mesh_check(GearGeometry(z=20, m=2, k_tip=-0.3), GearGeometry(z=40, m=2, k_tip=-0.3))
assert not bad.ok

## References

- ISO 21771:2007 — *Gears — Cylindrical involute gears and gear pairs — Concepts and geometry*. Signed tooth counts for internal gears, profile shift, operating pressure angle.
- ISO 53:1998 — *Standard basic rack tooth profile* (profile A: α = 20°, h_aP = 1.0 m, h_fP = 1.25 m, ρ_fP = 0.38 m).
- ISO 54:1996 — *Modules*.
- DIN 3960:1987 — *Definitions, parameters and equations for involute cylindrical gears*: span and over-ball measurements.
- Linke, H. (2010). *Stirnradverzahnung* (2nd ed.). Hanser — interference checks for internal gear pairs.